# Steam Market Analytics — Gold Core & Serving Materialization

This notebook materializes the Gold layer of the Steam analytics platform.

The architecture separates governance-oriented core tables from
BI-oriented serving marts to preserve analytical correctness while
supporting high-performance dashboard workloads.

The Gold Core layer maintains a strict 1-row-per-game grain through
`fact_games`, while many-to-many relationships such as genres and
publishers are isolated into dedicated bridge tables.

The Gold Serving layer intentionally denormalizes specific analytical
axes (genre, publisher) and computes fractional allocations to simplify
aggregations inside Power BI / Databricks SQL dashboards without
introducing revenue duplication.

This design follows a hybrid Lakehouse modeling strategy:
- normalized canonical facts in Core
- denormalized marts for downstream analytics consumption

In [0]:
catalog = "steam"

silver_schema = "silver"
gold_schema = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")


DataFrame[]

# Exploratory Analytics

Exploration is intentionally performed directly on Silver tables before productizing business logic into Gold assets.

This phase identifies:
- market structure
- pricing dynamics
- publisher concentration
- genre profitability
- review behavior
- release trends


## Publisher Market Share

In [0]:
%sql

SELECT
    publisher,
    COUNT(*) AS games_released
FROM steam.silver.steam_games
GROUP BY publisher
ORDER BY games_released DESC
LIMIT 20

publisher,games_released
Big Fish Games,422
8floor,202
SEGA,162
Strategy First,151
Square Enix,140
Choice of Games,140
null,134
Sekai Project,132
HH-Games,132
Ubisoft,126


Missing publishers are to be replaced with 'Unknown' to keep the dimensions consistent and avoid losing records in BI aggregations.

## Best Rated Games

To avoid rankings driven by very small sample sizes, only games with at least 500 total reviews are included.

This keeps the ratings more reliable while still retaining successful indie and mid-sized games such as Dorfromantik.

In [0]:
%sql

SELECT
    name,
    publisher,
    positive,
    negative,
    ROUND(positive_ratio, 3) AS positive_ratio
FROM steam.silver.steam_games
WHERE positive + negative >= 500
ORDER BY positive_ratio DESC
LIMIT 20

name,publisher,positive,negative,positive_ratio
Flowers -Le volume sur ete-,JAST USA,937,1,0.999
Aventura Copilului Albastru și Urât,Codrin Bradea,2203,14,0.994
Touhou Fuujinroku ~ Mountain of Faith.,"Mediascape Co., Ltd.",649,4,0.994
Aseprite,Igara Studio,11823,80,0.993
The Case of the Golden Idol,Playstack,909,6,0.993
A Short Hike,adamgryu,11645,87,0.993
Monster Prom 3: Monster Roadtrip,Beautiful Glitch,1052,8,0.993
Patrick's Parabox,Patrick Traynor,1489,11,0.993
Senren＊Banka,"HIKARI FIELD, NekoNyan Ltd.",10593,84,0.992
CULTIC,3D Realms,2021,16,0.992


## Release Trends

In [0]:
%sql

SELECT
    YEAR(release_date) AS release_year,
    COUNT(*) AS games_released
FROM steam.silver.steam_games
GROUP BY YEAR(release_date)
ORDER BY release_year

release_year,games_released
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


## Price Distribution

In [0]:
%sql

SELECT
    CASE
        WHEN price = 0 THEN 'Free To Play'
        WHEN price < 10 THEN 'Under 10$'
        WHEN price < 30 THEN '10$ - 30$'
        WHEN price < 60 THEN '30$ - 60$'
        ELSE '60$+'
    END AS price_bucket,
    COUNT(*) AS games_count
FROM steam.silver.steam_games
GROUP BY price_bucket
ORDER BY games_count DESC

price_bucket,games_count
Under 10$,35910
10$ - 30$,10802
Free To Play,7711
30$ - 60$,1042
60$+,122


## Genre Profitability

In [0]:
%sql

SELECT
    genre,
    COUNT(*) AS games_count,
    ROUND(AVG(positive_ratio), 2) AS avg_positive_ratio,
    ROUND(AVG(price * owners_midpoint), 0) AS avg_estimated_revenue
FROM steam.silver.steam_games_genres
GROUP BY genre
HAVING COUNT(*) >= 50
ORDER BY avg_estimated_revenue DESC

genre,games_count,avg_positive_ratio,avg_estimated_revenue
Massively Multiplayer,1459,0.63,4064536.0
RPG,9526,0.73,2847720.0
Action,23727,0.73,2463703.0
Strategy,10878,0.72,1845586.0
Adventure,21402,0.74,1729270.0
Simulation,10819,0.69,1721177.0
Photo Editing,105,0.7,1664437.0
Racing,2150,0.7,1266735.0
Sports,2661,0.7,1183701.0
Web Publishing,89,0.72,1177199.0


## 1. Core Dimension Modeling (Conformed Dimensions)

Dimensions are extracted independently from the Silver layer
to provide reusable analytical axes across downstream marts.

SHA-256 surrogate keys are generated to guarantee stable joins
across distributed Spark workloads.


## dim_publishers

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.dim_publishers AS

SELECT DISTINCT

    sha2(
        COALESCE(
            NULLIF(TRIM(publisher), ''),
            'Unknown'
        ),
        256
    ) AS publisher_key,

    COALESCE(
        NULLIF(TRIM(publisher), ''),
        'Unknown'
    ) AS publisher_name,

    current_timestamp() AS dw_inserted_at

FROM steam.silver.steam_games

num_affected_rows,num_inserted_rows


#### Quick data quality validation to identify:
 - missing publisher coverage
 - malformed publisher labels
 - unexpected cardinality inflation

In [0]:
%sql

SELECT

    dp.publisher_name,

    COUNT(*) AS total_games

FROM steam.gold.bridge_game_publishers bgp

INNER JOIN steam.gold.dim_publishers dp
    ON bgp.publisher_key = dp.publisher_key

GROUP BY dp.publisher_name

ORDER BY total_games DESC

publisher_name,total_games
Big Fish Games,423
8floor,202
SEGA,162
Strategy First,151
Square Enix,140
Choice of Games,140
Unknown,134
Sekai Project,132
HH-Games,132
Laush Studio,126


## dim_genres

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.dim_genres AS

SELECT
    sha2(TRIM(genre), 256) AS genre_key,
    genre AS genre_name,
    current_timestamp() AS dw_inserted_at

FROM (
    SELECT DISTINCT genre
    FROM steam.silver.steam_games_genres
    WHERE genre IS NOT NULL
      AND TRIM(genre) != ''
)

num_affected_rows,num_inserted_rows


## dim_dates

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.dim_dates AS

SELECT DISTINCT

    CAST(date_format(release_date, 'yyyyMMdd') AS INT) AS date_key,

    release_date AS full_date,

    YEAR(release_date) AS year,

    QUARTER(release_date) AS quarter,

    MONTH(release_date) AS month,

    CASE
        WHEN YEAR(release_date) IN (2020, 2021)
        THEN TRUE
        ELSE FALSE
    END AS covid_period

FROM steam.silver.steam_games
WHERE release_date IS NOT NULL

num_affected_rows,num_inserted_rows


## 2. Core Fact Layer

`fact_games` preserves the canonical business grain:
1 row = 1 Steam game (`appid`).

Financial and engagement metrics are intentionally stored only once
at the game grain to guarantee mathematically correct global
aggregations across the platform.

Genres and publishers are excluded from the fact table because
both dimensions can create many-to-many relationships.


In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.fact_games AS

SELECT

    sg.appid,

    CAST(date_format(sg.release_date, 'yyyyMMdd') AS INT) AS date_key,

    sg.name,

    sg.price,

    sg.discount,

    ROUND(
        sg.price * sg.owners_midpoint,
        0
    ) AS estimated_revenue,

    sg.positive,

    sg.negative,

    sg.positive_ratio,

    sg.owners_midpoint,

    sg.ccu,

    sg.windows,
    sg.mac,
    sg.linux,

    sg.english,
    sg.french,
    sg.german,
    sg.spanish,
    sg.japanese,
    sg.russian,
    sg.simplified_chinese,
    sg.traditional_chinese,
    sg.korean,
    sg.portuguese_brazil,
    sg.italian,
    sg.polish,
    sg.turkish,

    sg.required_age,

    current_timestamp() AS dw_inserted_at

FROM steam.silver.steam_games sg

num_affected_rows,num_inserted_rows


## 3. Many-to-Many Bridge Resolution

Steam games can simultaneously belong to multiple genres
and multiple publishers.

Example:
- Helldivers 2 → Action + Shooter
- Helldivers 2 → Sony + Arrowhead

Embedding these attributes directly into `fact_games`
would trigger a fan-out effect, artificially inflating 
revenue and ownership metrics during BI aggregations.

Bridge tables isolate these analytical axes while preserving
a strict game-level grain inside the core fact layer.

### Building the bridge_game_genres

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.bridge_game_genres AS

SELECT DISTINCT

    sgg.appid,

    sha2(TRIM(sgg.genre), 256) AS genre_key,

    current_timestamp() AS dw_inserted_at

FROM steam.silver.steam_games_genres sgg

num_affected_rows,num_inserted_rows


## Building the bridge_game_publishers

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.bridge_game_publishers AS

SELECT DISTINCT

    sg.appid,

    sha2(
        COALESCE(
            NULLIF(TRIM(sg.publisher), ''),
            'Unknown'
        ),
        256
    ) AS publisher_key,

    current_timestamp() AS dw_inserted_at

FROM steam.silver.steam_games sg

num_affected_rows,num_inserted_rows


In [0]:
%sql

OPTIMIZE steam.gold.fact_games
ZORDER BY (date_key, appid)

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1285366), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1779626075213, 1779626075798, 8, 0, null, List(0, 0), null, 29, 29, 0, 0, null, null)"



## 4. Analytical Serving Layer (Data Marts)

Serving marts intentionally denormalize analytical dimensions
to optimize dashboard workloads and simplify downstream queries.

Revenue and ownership metrics are fractionally allocated across
genres and publishers using allocation coefficients computed
from bridge cardinality.

This approach avoids fan traps inside Power BI while preserving
accurate global totals inside the canonical core layer.

## mart_genre_performance

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.mart_genre_performance AS

WITH genre_counts AS (

    SELECT

        appid,

        COUNT(*) AS genre_count

    FROM steam.gold.bridge_game_genres

    GROUP BY appid
)

SELECT

    fg.appid,

    bgg.genre_key,

    fg.date_key,

    fg.name,

    fg.price,

    fg.discount,

    fg.estimated_revenue,

    ROUND(
        fg.estimated_revenue / gc.genre_count,
        2
    ) AS allocated_revenue,

    ROUND(
        fg.owners_midpoint / gc.genre_count,
        2
    ) AS allocated_owners,

    ROUND(
        fg.positive / gc.genre_count,
        2
    ) AS allocated_positive_reviews,

    ROUND(
        fg.negative / gc.genre_count,
        2
    ) AS allocated_negative_reviews,

    fg.positive_ratio,

    fg.ccu,

    fg.windows,
    fg.mac,
    fg.linux,

    fg.required_age,

    current_timestamp() AS dw_inserted_at

FROM steam.gold.fact_games fg

INNER JOIN steam.gold.bridge_game_genres bgg
    ON fg.appid = bgg.appid

INNER JOIN genre_counts gc
    ON fg.appid = gc.appid

num_affected_rows,num_inserted_rows


## mart_publisher_performance

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.mart_publisher_performance AS

WITH publisher_counts AS (

    SELECT

        appid,

        COUNT(*) AS publisher_count

    FROM steam.gold.bridge_game_publishers

    GROUP BY appid
)

SELECT

    fg.appid,

    bgp.publisher_key,

    fg.date_key,

    fg.name,

    fg.price,

    fg.discount,

    fg.estimated_revenue,

    ROUND(
        fg.estimated_revenue / pc.publisher_count,
        2
    ) AS allocated_revenue,

    ROUND(
        fg.owners_midpoint / pc.publisher_count,
        2
    ) AS allocated_owners,

    ROUND(
        fg.positive / pc.publisher_count,
        2
    ) AS allocated_positive_reviews,

    ROUND(
        fg.negative / pc.publisher_count,
        2
    ) AS allocated_negative_reviews,

    fg.positive_ratio,

    fg.ccu,

    fg.windows,
    fg.mac,
    fg.linux,

    fg.required_age,

    current_timestamp() AS dw_inserted_at

FROM steam.gold.fact_games fg

INNER JOIN steam.gold.bridge_game_publishers bgp
    ON fg.appid = bgp.appid

INNER JOIN publisher_counts pc
    ON fg.appid = pc.appid

num_affected_rows,num_inserted_rows


## publisher_genre_affinity

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.publisher_genre_affinity AS

SELECT

    dpub.publisher_name,

    dgen.genre_name,

    COUNT(DISTINCT mgp.appid) AS total_games,

    ROUND(
        SUM(mgp.allocated_revenue),
        0
    ) AS total_allocated_revenue,

    ROUND(
        AVG(mgp.positive_ratio),
        2
    ) AS avg_positive_ratio

FROM steam.gold.mart_genre_performance mgp

INNER JOIN steam.gold.bridge_game_publishers bgp
    ON mgp.appid = bgp.appid

INNER JOIN steam.gold.dim_publishers dpub
    ON bgp.publisher_key = dpub.publisher_key

INNER JOIN steam.gold.dim_genres dgen
    ON mgp.genre_key = dgen.genre_key

GROUP BY

    dpub.publisher_name,
    dgen.genre_name

num_affected_rows,num_inserted_rows


## genre_platform_summary

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.genre_platform_summary AS

SELECT

    dgen.genre_name,

    COUNT(DISTINCT mgp.appid) AS total_games,

    ROUND(
        SUM(mgp.allocated_revenue),
        0
    ) AS total_allocated_revenue,

    ROUND(
        AVG(mgp.positive_ratio),
        2
    ) AS avg_positive_ratio,

    SUM(
        CASE WHEN mgp.windows THEN 1 ELSE 0 END
    ) AS windows_games,

    SUM(
        CASE WHEN mgp.mac THEN 1 ELSE 0 END
    ) AS mac_games,

    SUM(
        CASE WHEN mgp.linux THEN 1 ELSE 0 END
    ) AS linux_games

FROM steam.gold.mart_genre_performance mgp

INNER JOIN steam.gold.dim_genres dgen
    ON mgp.genre_key = dgen.genre_key

GROUP BY dgen.genre_name

num_affected_rows,num_inserted_rows


## market_trends

In [0]:
%sql

CREATE OR REPLACE TABLE steam.gold.market_trends AS

SELECT

    dd.year,

    dd.quarter,

    COUNT(DISTINCT fg.appid) AS total_games_released,

    ROUND(
        SUM(fg.estimated_revenue),
        0
    ) AS total_estimated_revenue,

    ROUND(
        AVG(fg.price),
        2
    ) AS avg_game_price,

    ROUND(
        AVG(fg.positive_ratio),
        2
    ) AS avg_positive_ratio,

    ROUND(
        AVG(fg.ccu),
        0
    ) AS avg_ccu

FROM steam.gold.fact_games fg

INNER JOIN steam.gold.dim_dates dd
    ON fg.date_key = dd.date_key

GROUP BY

    dd.year,
    dd.quarter

num_affected_rows,num_inserted_rows


In [0]:
%sql

OPTIMIZE steam.gold.mart_genre_performance
ZORDER BY (genre_key, date_key)

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 3478113), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1779626100235, 1779626100734, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, null, null)"


In [0]:
%sql

OPTIMIZE steam.gold.mart_publisher_performance
ZORDER BY (publisher_key, date_key)

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2677680), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1779626102729, 1779626103329, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, null, null)"


# Validation Queries


In [0]:
%sql

SELECT COUNT(*) AS fact_rows FROM steam.gold.fact_games

fact_rows
55587


In [0]:
%sql

SELECT * FROM steam.gold.fact_games LIMIT 10

appid,date_key,name,price,discount,estimated_revenue,positive,negative,positive_ratio,owners_midpoint,ccu,windows,mac,linux,english,french,german,spanish,japanese,russian,simplified_chinese,traditional_chinese,korean,portuguese_brazil,italian,polish,turkish,required_age,dw_inserted_at
1000540,20190321,Tactical Control,0.0,0,0.0,53,12,0.8154,35000,0,true,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1001500,20190111,Chronicon Apocalyptica,5.99,0,59900.0,6,2,0.75,10000,0,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1002200,20190415,Vasilis,8.99,0,314650.0,10,6,0.625,35000,0,true,true,false,true,false,false,false,false,true,false,false,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1004000,20190126,Card Brawl,1.99,0,19900.0,20,9,0.6897,10000,0,true,false,false,true,false,false,false,false,true,false,false,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1015350,20190227,Your Home,0.99,0,9900.0,30,17,0.6383,10000,0,true,false,false,true,false,false,false,false,true,false,false,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1020600,20190215,Extreme Truck Simulator,9.99,0,99900.0,26,21,0.5532,10000,0,true,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1022150,20200417,Bios Ex - Yami no Wakusei,9.99,0,99900.0,8,3,0.7273,10000,0,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1022530,20190403,SUBARACITY,4.99,0,49900.0,10,4,0.7143,10000,0,true,false,false,true,false,false,false,true,false,false,true,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1022630,20190313,Tankorama,6.99,0,69900.0,4,0,1.0,10000,0,true,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,0,2026-05-24T12:34:25.042Z
1023390,20200307,People,4.99,0,49900.0,9,9,0.5,10000,0,true,false,false,true,true,true,true,true,true,true,true,true,true,true,true,true,0,2026-05-24T12:34:25.042Z


In [0]:
%sql

DESCRIBE EXTENDED steam.gold.fact_games

col_name,data_type,comment
appid,bigint,null
date_key,int,null
name,string,null
price,double,null
discount,int,null
estimated_revenue,double,null
positive,bigint,null
negative,bigint,null
positive_ratio,double,null
owners_midpoint,int,null


In [0]:
%sql

SHOW TABLES IN steam.gold

database,tableName,isTemporary
gold,bridge_game_genres,false
gold,bridge_game_publishers,false
gold,dim_dates,false
gold,dim_genres,false
gold,dim_publishers,false
gold,fact_games,false
gold,genre_platform_summary,false
gold,market_trends,false
gold,mart_genre_performance,false
gold,mart_publisher_performance,false
